# Fase 3 — MLP e Backpropagation

## 🎯 Objetivo
Entender como uma rede com múltiplas camadas aprende, desde o fluxo de dados (forward pass) até a atualização dos pesos (backward pass).

Ao final deste notebook você será capaz de:
- Explicar o que é uma **camada oculta** e por que ela aumenta o poder do modelo
- Implementar um **forward pass** camada por camada com números
- Calcular a **CrossEntropyLoss** manualmente
- Derivar e implementar o **backpropagation** passo a passo
- Entender o ciclo completo: `zero_grad → forward → loss → backward → step`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
print("Bibliotecas carregadas!")

## Camada Oculta: Por que ela aumenta o poder?

Uma rede sem camada oculta só consegue criar fronteiras **lineares**.  
A camada oculta cria uma **representação intermediária** dos dados — ela transforma o espaço de entrada em um novo espaço onde as classes ficam mais fáceis de separar.

**Arquitetura do notebook original:**
```
Entrada (2) → Linear(2→4) → ReLU → Linear(4→3) → Saída (3 classes)
```

Cada neurônio da camada oculta aprende uma "feature" diferente dos dados.

In [ ]:
# Visualizando como a camada oculta transforma o espaço
np.random.seed(42)
N = 60
c1 = np.random.randn(N, 2) * 0.12 + [0.90, 0.90]
c2 = np.random.randn(N, 2) * 0.12 + [0.60, 0.60]
c3 = np.random.randn(N, 2) * 0.12 + [0.30, 0.30]

X = np.vstack([c1, c2, c3])
Y = np.array([0]*N + [1]*N + [2]*N)
paleta = ['red', 'green', 'blue']

# Pesos manuais da camada oculta (como no notebook original)
v  = (2.0**0.5) / 2
b1 = ((0.85)**2 + (0.85)**2)**0.5
b2 = ((0.55)**2 + (0.55)**2)**0.5

W1 = np.array([[v,v], [-v,-v], [v,v], [-v,-v]])
b_1 = np.array([-b1, b1, -b2, b2])

# Forward da camada oculta
Z1 = X @ W1.T + b_1                  # pré-ativação
H  = np.maximum(0, Z1)               # ReLU (camada oculta)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Espaço original
for cls in range(3):
    mask = Y == cls
    ax1.scatter(X[mask,0], X[mask,1], c=paleta[cls], alpha=0.7, s=25, label=f'Classe {cls+1}')
ax1.set_title('Espaço original (entrada)', fontsize=11)
ax1.set_xlabel('x₁'); ax1.set_ylabel('x₂')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Espaço transformado pela camada oculta (mostrando 2 das 4 dimensões)
for cls in range(3):
    mask = Y == cls
    ax2.scatter(H[mask,0], H[mask,1], c=paleta[cls], alpha=0.7, s=25, label=f'Classe {cls+1}')
ax2.set_title('Espaço transformado pela camada oculta
(neurônios 1 e 2 de 4)', fontsize=11)
ax2.set_xlabel('Neurônio 1'); ax2.set_ylabel('Neurônio 2')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('A camada oculta transforma o espaço de entrada', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## Forward Pass: Dados Fluindo pela Rede

O **forward pass** é o cálculo da predição da rede para uma entrada:

**Camada 1 (oculta):**
$$\mathbf{z}^{(1)} = W^{(1)} \mathbf{x} + \mathbf{b}^{(1)}$$
$$\mathbf{h} = \text{ReLU}(\mathbf{z}^{(1)}) = \max(0, \mathbf{z}^{(1)})$$

**Camada 2 (decisão):**
$$\mathbf{z}^{(2)} = W^{(2)} \mathbf{h} + \mathbf{b}^{(2)}$$
$$\hat{\mathbf{y}} = \text{Softmax}(\mathbf{z}^{(2)}) = \frac{e^{z_k^{(2)}}}{\sum_j e^{z_j^{(2)}}}$$

O **Softmax** converte os logits em probabilidades que somam 1 — uma probabilidade por classe.

In [ ]:
# === Forward pass completo com números ===
# Um exemplo com um único ponto de entrada
x_sample = np.array([0.85, 0.85])   # ponto da classe 1

# === Pesos da camada oculta (4 neurônios) ===
W1 = np.array([[ 0.72,  0.72],
               [-0.72, -0.72],
               [ 0.72,  0.72],
               [-0.72, -0.72]])
b1 = np.array([-1.20,  1.20, -0.78,  0.78])

# === Pesos da camada de decisão (3 neurônios) ===
np.random.seed(10)
W2 = np.random.randn(3, 4) * 0.5
b2 = np.zeros(3)

print("=" * 50)
print("FORWARD PASS — passo a passo")
print("=" * 50)
print(f"
Entrada: x = {x_sample}")
print(f"
--- Camada Oculta ---")
z1 = W1 @ x_sample + b1
print(f"Pré-ativação z1 = W1·x + b1 = {z1.round(4)}")
h  = np.maximum(0, z1)    # ReLU
print(f"Ativação    h   = ReLU(z1)  = {h.round(4)}")

print(f"
--- Camada de Decisão ---")
z2 = W2 @ h + b2
print(f"Logits z2 = W2·h + b2 = {z2.round(4)}")
exp_z2 = np.exp(z2 - np.max(z2))  # subtração para estabilidade numérica
softmax = exp_z2 / exp_z2.sum()
print(f"Softmax(z2) = {softmax.round(4)}")
print(f"
Predição: Classe {np.argmax(softmax) + 1} com probabilidade {softmax.max():.2%}")

## CrossEntropyLoss: Medindo o Erro de Classificação

A **CrossEntropyLoss** mede quão "surpreendido" o modelo está em relação à resposta correta.

Para a classe verdadeira $c$:
$$L = -\log(\hat{y}_c)$$

Para um batch de $N$ exemplos:
$$L = -\frac{1}{N}\sum_{i=1}^{N} \log(\hat{y}_{i,c_i})$$

**Intuição:**
- Se a rede prediz 99% para a classe correta → $L = -\log(0.99) \approx 0.01$ (loss baixo ✓)
- Se a rede prediz 1% para a classe correta  → $L = -\log(0.01) \approx 4.6$  (loss alto ✗)

**Por que não usar MSE para classificação?**  
MSE trata classes como números contínuos (classe 2 está "entre" 1 e 3), o que não faz sentido. CrossEntropy trata cada classe como evento probabilístico independente.

In [ ]:
# === CrossEntropyLoss: visualização e cálculo ===
# Probabilidade predita para a classe correta
p_correto = np.linspace(0.01, 0.999, 300)
loss = -np.log(p_correto)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(p_correto, loss, 'b-', linewidth=2.5)
ax1.axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='p=0.5 (incerto)')
ax1.axhline(0.69, color='red', linestyle='--', alpha=0.7, label='L=-log(0.5)≈0.69')
ax1.set_xlabel('Probabilidade predita para classe correta')
ax1.set_ylabel('CrossEntropy Loss')
ax1.set_title('Quanto maior a incerteza, maior a perda')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Exemplo numérico com 3 amostras
Y_true = np.array([0, 1, 2])            # classes verdadeiras
logits = np.array([[2.0, 0.5, 0.1],     # boa predição (classe 0)
                   [0.1, 1.8, 0.3],     # boa predição (classe 1)
                   [0.8, 0.4, 0.1]])    # má predição  (classe 2)

def softmax_batch(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy(logits, y_true):
    probs = softmax_batch(logits)
    N = len(y_true)
    return -np.mean(np.log(probs[np.arange(N), y_true] + 1e-9))

probs = softmax_batch(logits)
total_loss = cross_entropy(logits, Y_true)

nomes = ['Amostra 1
(boa pred.)', 'Amostra 2
(boa pred.)', 'Amostra 3
(má pred.)']
losses_ind = [-np.log(probs[i, Y_true[i]]) for i in range(3)]
bars = ax2.bar(nomes, losses_ind, color=['green', 'green', 'red'], alpha=0.7)
ax2.axhline(total_loss, color='black', linestyle='--', label=f'Loss médio = {total_loss:.3f}')
ax2.set_ylabel('CrossEntropy Loss individual')
ax2.set_title('Loss por amostra — má predição gera loss alto')
ax2.legend(); ax2.grid(True, alpha=0.3)
for bar, val in zip(bars, losses_ind):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.3f}', ha='center', fontsize=11)

plt.tight_layout(); plt.show()
print(f"Loss total (média): {total_loss:.4f}")

## Backpropagation: Calculando os Gradientes

O **backpropagation** usa a regra da cadeia do cálculo para calcular $\frac{\partial L}{\partial w}$ para cada peso da rede, propagando o erro da saída até a entrada.

### Regra da cadeia:
$$\frac{\partial L}{\partial W^{(1)}} = \frac{\partial L}{\partial \mathbf{z}^{(2)}} \cdot \frac{\partial \mathbf{z}^{(2)}}{\partial \mathbf{h}} \cdot \frac{\partial \mathbf{h}}{\partial \mathbf{z}^{(1)}} \cdot \frac{\partial \mathbf{z}^{(1)}}{\partial W^{(1)}}$$

**De trás para frente (daí o nome):**

| Passo | Gradiente | Fórmula |
|---|---|---|
| 1 | $\delta^{(2)}$ | $\hat{\mathbf{y}} - \mathbf{y}_{\text{onehot}}$ (Softmax + CrossEntropy) |
| 2 | $\nabla_{W^{(2)}}L$ | $\delta^{(2)} \cdot \mathbf{h}^T$ |
| 3 | $\delta^{(1)}$ | $(W^{(2)T} \cdot \delta^{(2)}) \odot \text{ReLU}'(\mathbf{z}^{(1)})$ |
| 4 | $\nabla_{W^{(1)}}L$ | $\delta^{(1)} \cdot \mathbf{x}^T$ |

In [ ]:
# === Backpropagation manual em um MLP 2→4→3 ===
np.random.seed(42)
N_tr, N_feat, N_hid, N_cls = 150, 2, 4, 3

# Dados
c1 = np.random.randn(N_tr//3, 2) * 0.12 + [0.90, 0.90]
c2 = np.random.randn(N_tr//3, 2) * 0.12 + [0.60, 0.60]
c3 = np.random.randn(N_tr//3, 2) * 0.12 + [0.30, 0.30]
X = np.vstack([c1, c2, c3])
Y = np.array([0]*(N_tr//3) + [1]*(N_tr//3) + [2]*(N_tr//3))

# One-hot encoding
Y_oh = np.zeros((N_tr, N_cls))
Y_oh[np.arange(N_tr), Y] = 1

# Xavier initialization
np.random.seed(0)
W1 = np.random.randn(N_hid, N_feat) * np.sqrt(2.0 / (N_feat + N_hid))
b1 = np.zeros(N_hid)
W2 = np.random.randn(N_cls, N_hid) * np.sqrt(2.0 / (N_hid + N_cls))
b2 = np.zeros(N_cls)

lr = 0.5
epochs = 200
losses, accs = [], []

def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

for epoch in range(epochs):
    # === FORWARD PASS ===
    Z1    = X @ W1.T + b1        # (N, 4)
    H     = np.maximum(0, Z1)    # ReLU
    Z2    = H @ W2.T + b2        # (N, 3) — logits
    Y_hat = softmax(Z2)           # (N, 3) — probabilidades

    loss = -np.mean(np.sum(Y_oh * np.log(Y_hat + 1e-9), axis=1))
    acc  = np.mean(Y_hat.argmax(axis=1) == Y)
    losses.append(loss); accs.append(acc)

    # === BACKWARD PASS ===
    # Gradiente da loss em relação aos logits (Softmax + CrossEntropy)
    dZ2 = (Y_hat - Y_oh) / N_tr                # (N, 3)

    # Gradientes da camada 2
    dW2 = dZ2.T @ H                             # (3, 4)
    db2 = dZ2.sum(axis=0)                       # (3,)

    # Propagar o gradiente para a camada oculta
    dH  = dZ2 @ W2                              # (N, 4)
    dZ1 = dH * (Z1 > 0).astype(float)          # ReLU' (gradiente)

    # Gradientes da camada 1
    dW1 = dZ1.T @ X                             # (4, 2)
    db1 = dZ1.sum(axis=0)                       # (4,)

    # Atualização dos pesos (SGD)
    W2 -= lr * dW2;  b2 -= lr * db2
    W1 -= lr * dW1;  b1 -= lr * db1

print(f"Época {epochs}: Loss={losses[-1]:.4f}  Acurácia={accs[-1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(losses, 'r-', lw=2, label='Loss'); ax1.set_title('Curva de Loss')
ax1_b = ax1.twinx()
ax1_b.plot(accs, 'b--', lw=2, label='Acurácia')
ax1.set_xlabel('Época'); ax1.set_ylabel('Loss', color='red')
ax1_b.set_ylabel('Acurácia', color='blue'); ax1.grid(True, alpha=0.3)

# Fronteiras de decisão
xx, yy = np.meshgrid(np.linspace(0.1, 1.2, 200), np.linspace(0.1, 1.2, 200))
Xg = np.c_[xx.ravel(), yy.ravel()]
Zg = softmax(np.maximum(0, Xg @ W1.T + b1) @ W2.T + b2)
Pg = Zg.argmax(axis=1).reshape(xx.shape)
ax2.contourf(xx, yy, Pg, alpha=0.3, cmap='RdYlBu')
for cls, cor in enumerate(['red', 'green', 'blue']):
    mask = Y == cls
    ax2.scatter(X[mask,0], X[mask,1], c=cor, alpha=0.7, s=25, label=f'Classe {cls+1}')
ax2.set_title('Regiões de Decisão (MLP manual)'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()